# Assignment 2

In this assigment, we will work with the *Forest Fire* data set. Please download the data from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/162/forest+fires). Extract the data files into the subdirectory: `../data/fires/` (relative to `./05_src/`).

## Objective

+ The model objective is to predict the area affected by forest fires given the features set. 
+ The objective of this exercise is to assess your ability to construct and evaluate model pipelines.
+ Please note: the instructions are not meant to be 100% prescriptive, but instead they are a set of minimum requirements. If you find predictive performance gains by applying additional steps, by all means show them. 

## Variable Description

From the description file contained in the archive (`forestfires.names`), we obtain the following variable descriptions:

1. X - x-axis spatial coordinate within the Montesinho park map: 1 to 9
2. Y - y-axis spatial coordinate within the Montesinho park map: 2 to 9
3. month - month of the year: "jan" to "dec" 
4. day - day of the week: "mon" to "sun"
5. FFMC - FFMC index from the FWI system: 18.7 to 96.20
6. DMC - DMC index from the FWI system: 1.1 to 291.3 
7. DC - DC index from the FWI system: 7.9 to 860.6 
8. ISI - ISI index from the FWI system: 0.0 to 56.10
9. temp - temperature in Celsius degrees: 2.2 to 33.30
10. RH - relative humidity in %: 15.0 to 100
11. wind - wind speed in km/h: 0.40 to 9.40 
12. rain - outside rain in mm/m2 : 0.0 to 6.4 
13. area - the burned area of the forest (in ha): 0.00 to 1090.84 









### Specific Tasks

+ Construct four model pipelines, out of combinations of the following components:

    + Preprocessors:

        - A simple processor that only scales numeric variables and recodes categorical variables.
        - A transformation preprocessor that scales numeric variables and applies a non-linear transformation.
    
    + Regressor:

        - A baseline regressor, which could be a [K-nearest neighbours model]() or a linear model like [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) or [Ridge Regressors](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ridge_regression.html).
        - An advanced regressor of your choice (e.g., Bagging, Boosting, SVR, etc.). TIP: select a tree-based method such that it does not take too long to run SHAP further below. 

+ Evaluate tune and evaluate each of the four model pipelines. 

    - Select a [performance metric](https://scikit-learn.org/stable/modules/linear_model.html) out of the following options: explained variance, max error, root mean squared error (RMSE), mean absolute error (MAE), r-squared.
    - *TIPS*: 
    
        * Out of the suggested metrics above, [some are correlation metrics, but this is a prediction problem](https://www.tmwr.org/performance#performance). Choose wisely (and don't choose the incorrect options.) 

+ Select the best-performing model and explain its predictions.

    - Provide local explanations.
    - Obtain global explanations and recommend a variable selection strategy.

+ Export your model as a pickle file.


You can work on the Jupyter notebook, as this experiment is fairly short (no need to use sacred). 

# Load the data

Place the files in the ../../05_src/data/fires/ directory and load the appropriate file. 

In [37]:
# Load the libraries as required.
import pandas as pd

In [38]:
# Load data
columns = [
    'coord_x', 'coord_y', 'month', 'day', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain', 'area' 
]
fires_dt = (pd.read_csv('../../05_src/data/fires/forestfires.csv', header = 0, names = columns))


# Get X and Y

Create the features data frame and target data.

In [39]:
X = fires_dt.drop(columns=['area', 'month', 'day'])

In [40]:
Y = fires_dt[['area']]

# Preprocessing

Create two [Column Transformers](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html), called preproc1 and preproc2, with the following guidelines:

- Numerical variables

    * (Preproc 1 and 2) Scaling: use a scaling method of your choice (Standard, Robust, Min-Max). 
    * Preproc 2 only: 
        
        + Choose a transformation for any of your input variables (or several of them). Evaluate if this transformation is convenient.
        + The choice of scaler is up to you.

- Categorical variables: 
    
    * (Preproc 1 and 2) Apply [one-hot encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) where appropriate.


+ The only difference between preproc1 and preproc2 is the non-linear transformation of the numerical variables.
    


### Preproc 1

Create preproc1 below.

+ Numeric: scaled variables, no other transforms.
+ Categorical: one-hot encoding.

In [51]:
# import libraries 
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PowerTransformer
from sklearn.model_selection import train_test_split, GridSearchCV, cross_validate
from sklearn.neighbors import KNeighborsRegressor

In [42]:
ctransform_simple= ColumnTransformer([
    ('numeric_simple', StandardScaler(), ['ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind','rain']),
    ('cat_simple', OneHotEncoder(), ['coord_x','coord_y']),
], remainder='passthrough')
ctransform_simple

,transformers,"[('numeric_simple', ...), ('cat_simple', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,copy,True
,with_mean,True
,with_std,True


### Preproc 2

Create preproc2 below.

+ Numeric: scaled variables, non-linear transformation to one or more variables.
+ Categorical: one-hot encoding.

In [43]:
ctransform_yj = ColumnTransformer([
    ('numeric_std', StandardScaler(), ['temp', 'rh', 'wind','rain']),
    ('cat_simple', OneHotEncoder(), ['coord_x','coord_y']),
    ('numeric_yj', PowerTransformer(method='yeo-johnson'), ['ffmc', 'dmc', 'dc', 'isi'])
], remainder='passthrough')

ctransform_yj

,transformers,"[('numeric_std', ...), ('cat_simple', ...), ...]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,copy,True
,with_mean,True
,with_std,True


## Model Pipeline


Create a [model pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html): 

+ Add a step labelled `preprocessing` and assign the Column Transformer from the previous section.
+ Add a step labelled `regressor` and assign a regression model to it. 

## Regressor

+ Use a regression model to perform a prediction. 

    - Choose a baseline regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Choose a more advance regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Both model choices are up to you, feel free to experiment.

In [44]:
# Pipeline A = preproc1 + baseline
pipe_knn = Pipeline([
    ('preprocess1', ctransform_simple),
    ('knn', KNeighborsRegressor())
])
pipe_knn

,steps,"[('preprocess1', ...), ('knn', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric_simple', ...), ('cat_simple', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [ ]:
pipe_knn.get_params

In [48]:
# Pipeline B = preproc2 + baseline
pipe_knn_yj = Pipeline([
    ('preprocess2', ctransform_yj),
    ('knn', KNeighborsRegressor(n_neighbors=5))
])
pipe_knn_yj

,steps,"[('preprocess2', ...), ('knn', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric_std', ...), ('cat_simple', ...), ...]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [ ]:
# Pipeline C = preproc1 + advanced model
# I am going to skip this part because I don't know how to apply a more advanced regressor 

In [ ]:
# Pipeline D = preproc2 + advanced model

    

# Tune Hyperparams

+ Perform GridSearch on each of the four pipelines. 
+ Tune at least one hyperparameter per pipeline.
+ Experiment with at least four value combinations per pipeline.

In [47]:
# split data into train and test 
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, random_state = 1)

# use cross-val to determine optimal K
param_grid = {
    "knn__n_neighbors": [1, 50, 100, 150, 200, 250, 300]
}

fire_gridsearch = GridSearchCV(
    estimator=pipe_knn,
    param_grid=param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error",
)

fire_gridsearch.fit(X_train,Y_train)
fire_gridsearch.best_params_
fire_gridsearch.best_estimator_

,steps,"[('preprocess1', ...), ('knn', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric_simple', ...), ('cat_simple', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [ ]:
fire_simple_dict = cross_validate(pipe_knn, X_train, Y_train, cv = 5, scoring = "neg_root_mean_squared_error")
fire_simple = pd.DataFrame(fire_simple_dict).assign(experiment = 1) 
fire_simple

,fit_time,score_time,test_score,experiment
0,0.012547,0.006254,-121.952661,1
1,0.003936,0.003144,-50.296843,1
2,0.004476,0.002483,-29.628323,1
3,0.002751,0.002146,-35.375370,1
4,0.002825,0.002632,-63.211160,1


In [53]:
fire_simple.mean()

fit_time       0.005307
score_time     0.003332
test_score   -60.092871
experiment     1.000000
dtype: float64

In [49]:
fire_gridsearch_yj = GridSearchCV(
    estimator=pipe_knn_yj,
    param_grid=param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error",
)

fire_gridsearch_yj.fit(X_train,Y_train)
fire_gridsearch_yj.best_params_
fire_gridsearch_yj.best_estimator_

,steps,"[('preprocess2', ...), ('knn', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric_std', ...), ('cat_simple', ...), ...]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [54]:
fire_yj_dict = cross_validate(pipe_knn_yj, X_train, Y_train, cv = 5, scoring = "neg_root_mean_squared_error")
fire_yj = pd.DataFrame(fire_yj_dict).assign(experiment = 2) 
fire_yj

,fit_time,score_time,test_score,experiment
0,0.020221,0.004420,-121.586267,2
1,0.011413,0.002812,-55.310954,2
2,0.006526,0.002282,-52.675969,2
3,0.005558,0.002059,-35.515887,2
4,0.005470,0.002016,-62.651938,2


In [55]:
fire_yj.mean()

fit_time       0.009838
score_time     0.002718
test_score   -65.548203
experiment     2.000000
dtype: float64

# Evaluate

+ Which model has the best performance?

Our pipe_knn has better performance because the root mean square error is smaller (60.1)

# Export

+ Save the best performing model to a pickle file.

In [56]:
import pickle
import os

os.makedirs("./models", exist_ok=True)
with open('./models/fire_knn_grid_search.pkl', 'wb') as f: #wb = write binary 
    pickle.dump(fire_gridsearch.best_estimator_, f)

# Explain

+ Use SHAP values to explain the following only for the best-performing model:

    - Select an observation in your test set and explain which are the most important features that explain that observation's specific prediction.

    - In general, across the complete training set, which features are the most and least important.

+ If you were to remove features from the model, which ones would you remove? Why? How would you test that these features are actually enhancing model performance?

In [71]:
X_train

,coord_x,coord_y,ffmc,dmc,dc,isi,temp,rh,wind,rain
135,3,5,93.5,139.4,594.2,20.3,17.6,52,5.8,0.0
218,4,5,92.9,133.3,699.6,9.2,19.4,19,1.3,0.0
119,3,4,93.0,75.3,466.6,7.7,19.6,36,3.1,0.0
463,6,5,75.1,4.4,16.2,1.9,4.6,82,6.3,0.0
42,4,4,94.8,108.3,647.1,17.0,16.6,54,5.4,0.0
...,...,...,...,...,...,...,...,...,...,...
129,2,5,92.6,46.5,691.8,8.8,15.4,35,0.9,0.0
144,2,5,95.5,99.9,513.3,13.2,23.8,32,5.4,0.0
72,5,4,91.7,33.3,77.5,9.0,15.6,25,6.3,0.0
235,8,6,91.4,142.4,601.4,10.6,19.6,41,5.8,0.0


In [69]:
X_test

,coord_x,coord_y,ffmc,dmc,dc,isi,temp,rh,wind,rain
270,2,2,92.1,152.6,658.2,14.3,21.8,56,3.1,0.0
90,6,5,90.2,96.9,624.2,8.9,14.7,59,5.8,0.0
133,4,6,93.7,80.9,685.2,17.9,17.6,42,3.1,0.0
221,3,4,93.3,141.2,713.9,13.9,18.6,49,3.6,0.0
224,7,4,90.1,82.9,735.7,6.2,15.4,57,4.5,0.0
...,...,...,...,...,...,...,...,...,...,...
438,2,5,93.7,231.1,715.1,8.4,23.6,53,4.0,0.0
11,7,5,92.8,73.2,713.0,22.6,19.3,38,4.0,0.0
358,6,3,92.5,122.0,789.7,10.2,19.7,39,2.7,0.0
92,8,6,92.3,85.3,488.0,14.7,20.8,32,6.3,0.0


In [ ]:
pipe = fire_gridsearch.best_estimator_
pipe.named_steps['preprocess1'].transform(X_test)

ValueError: Found unknown categories [np.int64(8)] in column 1 during transform

In [65]:
import shap

data_transform = pipe.named_steps['preprocess1'].transform(X_test)

explainer = shap.explainers(
    pipe.named_steps['knn'], 
    data_transform,
    feature_names = pipe.named_steps['preprocess1'].get_feature_names_out())

shap_values = explainer(data_transform)

ValueError: Found unknown categories [np.int64(8)] in column 1 during transform

*(Answer here.)*

## Criteria

The [rubric](./assignment_2_rubric_clean.xlsx) contains the criteria for assessment.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-2`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_2.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at the `help` channel. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.

# Reference

Cortez,Paulo and Morais,Anbal. (2008). Forest Fires. UCI Machine Learning Repository. https://doi.org/10.24432/C5D88D.